# SaaS Support Desk with Metadata Filtering (Strands + AgentCore Memory)

## Introduction

This tutorial extends the base customer-support pattern to demonstrate **metadata-based retrieval** on AgentCore long-term memory. We model a realistic SaaS support desk where operational queries need to slice memory by escalation chain, resolution time, and product area — not just severity.

A query like *"find billing tickets that escalated to Tier-3 and took over two hours to resolve"* becomes a precise, multi-dimensional filter applied **before** the vector similarity search runs.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long-term memory with metadata filtering                                         |
| Agent type          | SaaS support desk                                                                |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                       |
| Tutorial components | Semantic memory with custom `metadataSchema`, indexed keys, metadata filters     |
| Example complexity  | Intermediate                                                                     |

### You'll learn to
- Create a memory resource with `indexedKeys` and a strategy-level `metadataSchema`
- Use custom `llmExtractionInstruction` (beyond `LATEST_VALUE`) for an **append-only STRINGLIST**
- Attach event-time metadata and let the FM infer the rest
- Retrieve memories with `EQUALS_TO`, `CONTAINS`, `GREATER_THAN`, and system-timestamp `AFTER` filters
- Measure the precision lift of filtered vs unfiltered retrieval
- Evolve the schema with `update_memory` (additive-only)

## Prerequisites
- Python 3.10+
- AWS credentials with `bedrock-agentcore` and `bedrock-agentcore-control` permissions
- A `memory_execution_role_arn` (custom extraction instructions require model invocation — [see IAM guide](../../../../../05-memory-security-patterns/01-memory-iam-policies/))
- Amazon Bedrock access to Anthropic Claude Haiku 4.5


## Step 1: Install Dependencies

In [ ]:
!pip install -qr requirements.txt

## Step 2: Imports and Configuration

In [ ]:
import logging
import time
import json
import uuid
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional

import boto3
from botocore.exceptions import ClientError

from strands import Agent, tool
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

from bedrock_agentcore.memory import MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.models import StringValue

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("saas-support-metadata")

USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

logger.info("Imports loaded")


In [ ]:
# Replace with your values
REGION = "us-west-2"
MEMORY_EXECUTION_ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<AgentCoreMemoryExecutionRole>"

CUSTOMER_ID = "customer-001"
SESSION_ID = f"support_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info(f"Region: {REGION}")
logger.info(f"Customer: {CUSTOMER_ID}")
logger.info(f"Session:  {SESSION_ID}")


## Step 3: Create the Memory Resource with Metadata Schema

We use the low-level `bedrock-agentcore-control` client to create the memory resource with:
- **4 indexed keys**: `escalation_chain`, `product_area`, `resolution_time_minutes`, `customer_health_score`
- A **`SemanticMemoryStrategy`** whose `metadataSchema` defines how each key is populated

Three things to call out in this schema:

1. **`escalation_chain` uses an append-only merge** — not `LATEST_VALUE`. As the ticket moves through handlers, each new handler is appended to the ordered list. This is very different from the default behavior and gives us a durable routing trail per ticket.
2. **`product_area` uses `allowedValues` validation** — the FM's output is constrained to a controlled vocabulary so downstream filters don't break on `billing` vs `Billing` vs `BILLING`.
3. **`customer_health_score` is schema-only (not indexed)** — the FM populates it on the record for reporting, but it can't be used in filters. This is the "enrichment" pattern: you get the value without burning an indexed-key slot.


In [ ]:
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

memory_name = "SaaSSupportDeskMetadataMemory"

indexed_keys = [
    {"key": "escalation_chain",        "type": "STRING_LIST"},
    {"key": "product_area",            "type": "STRING"},
    {"key": "resolution_time_minutes", "type": "NUMBER"},
    # Note: customer_health_score is intentionally NOT in indexedKeys.
]

metadata_schema = [
    {
        "key": "escalation_chain",
        "type": "STRING_LIST",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Ordered list of every handler (role or named agent) that touched the ticket "
                    "across the conversation. Examples: 'intake_bot', 'tier1_support', "
                    "'tier2_specialist', 'tier3_engineer', 'billing_specialist'."
                ),
                "llmExtractionInstruction": (
                    "Append each new handler to the existing ordered list as the ticket moves "
                    "through the desk. Preserve chronological order. Do NOT dedupe — if the ticket "
                    "returns to a previously-touched handler, append again to reflect the re-escalation."
                )
            }
        }
    },
    {
        "key": "product_area",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "The product surface the ticket concerns. Use billing for payments, invoicing, "
                    "refunds; auth for login, SSO, MFA issues; api for API errors and rate limits; "
                    "data_sync for integration and sync failures; ui for dashboard / UX issues; "
                    "infra for platform-wide outages."
                ),
                "llmExtractionInstruction": "LATEST_VALUE",
                "validation": {
                    "stringValidation": {
                        "allowedValues": ["billing", "auth", "api", "data_sync", "ui", "infra"]
                    }
                }
            }
        }
    },
    {
        "key": "resolution_time_minutes",
        "type": "NUMBER",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Total minutes elapsed between the first user message reporting the issue and "
                    "the handler's closing/resolution message. Return null/omit if the ticket has "
                    "not been resolved in this conversation."
                ),
                "llmExtractionInstruction": (
                    "Compute from conversation timestamps and explicit closure signals (e.g. "
                    "'resolved', 'closed', 'that fixed it'). If the ticket is still open, omit the value."
                )
            }
        }
    },
    {
        "key": "customer_health_score",
        "type": "NUMBER",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": (
                    "Aggregate customer frustration signal 0-100 (0 = furious, 100 = delighted) based "
                    "on tone, language, and the arc of the conversation. Used for reporting; not filterable."
                ),
                "llmExtractionInstruction": (
                    "Score the overall emotional arc of the conversation. Heavily weight the closing "
                    "turns. Clamp to [0, 100]."
                )
            }
        }
    },
]

create_params = {
    "name": memory_name,
    "eventExpiryDuration": 90,
    "memoryExecutionRoleArn": MEMORY_EXECUTION_ROLE_ARN,
    "indexedKeys": indexed_keys,
    "memoryStrategies": [
        {
            "semanticMemoryStrategy": {
                "name": "SupportSemanticWithMetadata",
                "description": "Support interactions with metadata-driven filtering",
                "namespaceTemplates": ["/support/{actorId}/"],
                "memoryRecordSchema": {
                    "metadataSchema": metadata_schema
                }
            }
        }
    ],
}

try:
    response = control_client.create_memory(**create_params)
    memory = response["memory"]
    memory_id = memory["id"]
    logger.info(f"✅ Created memory {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        existing = control_client.list_memories()["memories"]
        match = next((m for m in existing if m["name"] == memory_name), None)
        memory_id = match["id"]
        logger.info(f"ℹ️  Reusing existing memory {memory_id}")
    else:
        raise

# Capture the semantic strategy id for later use
memory_detail = control_client.get_memory(memoryId=memory_id)["memory"]
strategy_id = memory_detail["strategies"][0]["strategyId"]
namespace = f"/support/{CUSTOMER_ID}/"
logger.info(f"Strategy id: {strategy_id}")
logger.info(f"Namespace:   {namespace}")


## Step 4: Wait for Memory to be Active

Memory creation is asynchronous. Poll until the status is `ACTIVE` before writing events.


In [ ]:
def wait_active(memory_id, timeout_s=180):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        m = control_client.get_memory(memoryId=memory_id)["memory"]
        status = m["status"]
        logger.info(f"memory status: {status}")
        if status == "ACTIVE":
            return m
        if status == "FAILED":
            raise RuntimeError(f"Memory creation failed: {m.get('failureReason')}")
        time.sleep(5)
    raise TimeoutError("memory never reached ACTIVE")

wait_active(memory_id)


## Step 5: Session Manager and Hooks

Standard Strands hook plumbing — what's new is the `SupportDeskMetadataHooks` provider, which attaches **event-time metadata** (the current handler and inferred `product_area`) on every save. The other fields (`resolution_time_minutes`, `customer_health_score`, and the complete `escalation_chain`) are populated by the FM during extraction.


In [ ]:
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
customer_session = session_manager.create_memory_session(actor_id=CUSTOMER_ID, session_id=SESSION_ID)

class SupportDeskMetadataHooks(HookProvider):
    def __init__(self, session, default_handler: str = "tier1_support", default_area: str = "billing"):
        self.session = session
        self.current_handler = default_handler
        self.current_area = default_area
        self._retrieve_filters: Optional[List[Dict]] = None

    def set_context(self, handler: Optional[str] = None, product_area: Optional[str] = None,
                    retrieve_filters: Optional[List[Dict]] = None):
        if handler:
            self.current_handler = handler
        if product_area:
            self.current_area = product_area
        self._retrieve_filters = retrieve_filters

    def inject_context(self, event: MessageAddedEvent):
        msgs = event.agent.messages
        if not msgs or msgs[-1]["role"] != "user":
            return
        user_query = msgs[-1]["content"][0].get("text", "")
        if not user_query:
            return
        try:
            params = {
                "memoryId": memory_id,
                "namespace": namespace,
                "searchCriteria": {"searchQuery": user_query, "topK": 5},
            }
            if self._retrieve_filters:
                params["searchCriteria"]["metadataFilters"] = self._retrieve_filters
            data_client = boto3.client("bedrock-agentcore", region_name=REGION)
            resp = data_client.retrieve_memory_records(**params)
            records = resp.get("memoryRecordSummaries", [])
            if records:
                context = "\n".join(
                    f"- {r.get('content', {}).get('text', '')[:240]}" for r in records[:5]
                )
                event.agent.system_prompt = (
                    f"{event.agent.system_prompt}\n\nRelevant prior context:\n{context}"
                )
                logger.info(f"Injected {len(records)} prior memories (filters={bool(self._retrieve_filters)})")
        except Exception as e:
            logger.warning(f"Retrieval skipped: {e}")

    def save_turn(self, event: AfterInvocationEvent):
        msgs = event.agent.messages
        if len(msgs) < 2 or msgs[-1]["role"] != "assistant":
            return
        user_text = next((m["content"][0].get("text", "") for m in reversed(msgs) if m["role"] == "user"), "")
        asst_text = msgs[-1]["content"][0].get("text", "")
        if not user_text or not asst_text:
            return
        try:
            self.session.add_turns(
                messages=[ConversationalMessage(user_text, USER), ConversationalMessage(asst_text, ASSISTANT)],
                metadata={
                    "product_area":      StringValue.build(self.current_area),
                    "escalation_chain":  StringValue.build(self.current_handler),
                }
            )
            logger.info(f"Saved turn under handler='{self.current_handler}', area='{self.current_area}'")
        except Exception as e:
            logger.error(f"Save failed: {e}")

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(MessageAddedEvent, self.inject_context)
        registry.add_callback(AfterInvocationEvent, self.save_turn)

hooks = SupportDeskMetadataHooks(customer_session)
logger.info("Hook provider ready")


## Step 6: Seed a Realistic Ticket Lifecycle

We simulate four tickets spanning the full operational space — a long billing escalation that routes all the way to Tier-3, a quick auth self-service success, a mid-severity API rate-limit issue resolved at Tier-2, and a data-sync issue still open. Multiple product areas, multiple handlers, and variable resolution times give the filter corpus real shape.

Event-time metadata supplies the current handler and the product area. Everything else — the full `escalation_chain` list, `resolution_time_minutes`, and `customer_health_score` — is left for the FM to infer.


In [ ]:
def seed_ticket(turns, handler_sequence, product_area, backdate_hours=0):
    ts_base = datetime.now(timezone.utc) - timedelta(hours=backdate_hours)
    handlers = iter(handler_sequence)
    current = next(handlers)
    for i, (role, text) in enumerate(turns):
        msg = ConversationalMessage(text, USER if role == "user" else ASSISTANT)
        event_ts = ts_base + timedelta(minutes=i * 5)
        # Rotate handler every 2 turns
        if i > 0 and i % 2 == 0:
            try:
                current = next(handlers)
            except StopIteration:
                pass
        customer_session.add_turns(
            messages=[msg],
            metadata={
                "product_area":      StringValue.build(product_area),
                "escalation_chain":  StringValue.build(current),
            },
            event_timestamp=event_ts,
        )

# Ticket 1: Billing dispute, escalated all the way to Tier-3
ticket_1 = [
    ("user",      "My invoice for March shows $1200 but my plan is $400/mo. This is the third month in a row."),
    ("assistant", "I understand — I can see the discrepancy. Let me pull up your billing history."),
    ("user",      "I've already spoken to two people about this. I need someone senior."),
    ("assistant", "Escalating to our billing specialist. Holding while I transfer context."),
    ("user",      "Finally. Can you explain the overcharge?"),
    ("assistant", "After reviewing the ledger, the duplicate charge is from a metering bug affecting enterprise accounts. I'm looping in engineering for a full refund and root-cause."),
    ("user",      "Good. I expect this resolved today."),
    ("assistant", "Engineering confirmed the bug, refund of $2,400 processed, root-cause fix deploying this week. Ticket closed after 142 minutes end-to-end."),
]
seed_ticket(
    ticket_1,
    handler_sequence=["tier1_support", "billing_specialist", "tier3_engineer"],
    product_area="billing",
    backdate_hours=72,
)

# Ticket 2: Auth — quick self-service success
ticket_2 = [
    ("user",      "SSO stopped working for my team this morning."),
    ("assistant", "Usually a stale SAML assertion. Can you re-upload your IdP metadata?"),
    ("user",      "That fixed it. Thanks."),
    ("assistant", "Great — marking resolved, 8 minutes total."),
]
seed_ticket(ticket_2, ["tier1_support"], "auth", backdate_hours=48)

# Ticket 3: API rate-limit, resolved at Tier-2
ticket_3 = [
    ("user",      "We're getting 429s across the board since 10am."),
    ("assistant", "Reviewing your account's rate-limit metrics."),
    ("user",      "This is blocking production."),
    ("assistant", "Confirmed you hit the burst cap. Raising Tier-2 for temporary limit bump."),
    ("user",      "How long will the bump last?"),
    ("assistant", "Raised to 5x standard burst for 48 hours. Recommend upgrading to the Scale tier. Resolved in 95 minutes."),
]
seed_ticket(
    ticket_3,
    handler_sequence=["tier1_support", "tier2_specialist"],
    product_area="api",
    backdate_hours=24,
)

# Ticket 4: Data sync — still open (no resolution time)
ticket_4 = [
    ("user",      "Our Salesforce → warehouse sync has been stuck for 6 hours."),
    ("assistant", "Checking the connector status now."),
    ("user",      "Any ETA?"),
    ("assistant", "Connector is throwing auth errors against your SFDC instance. Can you confirm the OAuth token hasn't been rotated on your side?"),
]
seed_ticket(ticket_4, ["tier1_support"], "data_sync", backdate_hours=6)

logger.info("Seeded 4 tickets across billing, auth, api, data_sync")


## Step 7: Wait for Long-Term Memory Extraction

Extraction runs asynchronously (~1–2 minutes). Poll `list_memory_records` until the semantic strategy produces consolidated records.


In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=REGION)

def wait_for_records(expected_min=2, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        resp = data_client.list_memory_records(
            memoryId=memory_id, namespace=namespace, maxResults=50
        )
        recs = resp.get("memoryRecordSummaries", [])
        logger.info(f"records so far: {len(recs)}")
        if len(recs) >= expected_min:
            return recs
        time.sleep(15)
    raise TimeoutError(f"only found {len(recs)} records after {timeout_s}s")

records = wait_for_records(expected_min=3)
logger.info(f"✅ {len(records)} memory records extracted")


## Step 8: Inspect Extracted Metadata

Dump the full metadata on each consolidated record. You should see:
- `escalation_chain` as an **ordered list** — ticket 1 should contain all three handlers.
- `resolution_time_minutes` populated on the three resolved tickets, omitted on ticket 4 (still open).
- `product_area` constrained to the allowed vocabulary.
- `customer_health_score` populated on records but visible as a non-indexed schema value.


In [ ]:
def dump_records(recs):
    for i, r in enumerate(recs, 1):
        full = data_client.get_memory_record(memoryId=memory_id, memoryRecordId=r["memoryRecordId"])["memoryRecord"]
        print(f"\n=== Record #{i} ({full['memoryRecordId']}) ===")
        print(f"content: {full.get('content', {}).get('text', '')[:180]}")
        md_entries = full.get("metadata", {})
        for k, v in md_entries.items():
            print(f"  {k}: {v}")

dump_records(records)


## Step 9: Unfiltered Retrieval (Baseline)

The naive query: *"payment issues"* across the customer's full namespace. Semantic similarity is going to pull up everything touching billing or dollar amounts — including resolved tickets, sales-adjacent conversations, and anything the FM thought was payment-related.


In [ ]:
def retrieve(query, filters=None, top_k=10):
    search = {"searchQuery": query, "topK": top_k}
    if filters:
        search["metadataFilters"] = filters
    resp = data_client.retrieve_memory_records(
        memoryId=memory_id, namespace=namespace, searchCriteria=search
    )
    return resp.get("memoryRecordSummaries", [])

def show(results, label):
    print(f"\n--- {label}: {len(results)} results ---")
    for r in results:
        text = r.get("content", {}).get("text", "")[:150]
        score = r.get("score", 0)
        print(f"  [{score:.3f}] {text}")

baseline = retrieve("payment issues")
show(baseline, "UNFILTERED: 'payment issues'")


## Step 10: Filtered Retrieval — Operational Queries

Now the interesting ones. These are queries an operator *cannot* answer with similarity search alone.


In [ ]:
# Query 1: All billing-area memories ever touched by a Tier-3 engineer
f_billing_tier3 = [
    {"left": {"metadataKey": "product_area"}, "operator": "EQUALS_TO",
     "right": {"metadataValue": {"stringValue": "billing"}}},
    {"left": {"metadataKey": "escalation_chain"}, "operator": "CONTAINS",
     "right": {"metadataValue": {"stringValue": "tier3_engineer"}}},
]
r1 = retrieve("resolution steps", filters=f_billing_tier3)
show(r1, "FILTERED: billing + ever touched tier3_engineer")


In [ ]:
# Query 2: 'Stuck' tickets — resolution time > 120 minutes, any area
f_stuck = [
    {"left": {"metadataKey": "resolution_time_minutes"}, "operator": "GREATER_THAN",
     "right": {"metadataValue": {"numberValue": 120}}},
]
r2 = retrieve("resolution", filters=f_stuck)
show(r2, "FILTERED: resolution_time_minutes > 120")


In [ ]:
# Query 3: Recent auth issues — combine temporal + area filter
cutoff = (datetime.now(timezone.utc) - timedelta(days=7)).strftime("%Y-%m-%dT%H:%M:%SZ")
f_recent_auth = [
    {"left": {"metadataKey": "product_area"}, "operator": "EQUALS_TO",
     "right": {"metadataValue": {"stringValue": "auth"}}},
    {"left": {"metadataKey": "x-amz-agentcore-memory-createdAt"}, "operator": "AFTER",
     "right": {"metadataValue": {"dateTimeValue": cutoff}}},
]
r3 = retrieve("login problem", filters=f_recent_auth)
show(r3, f"FILTERED: product_area = auth AND createdAt AFTER {cutoff}")


In [ ]:
# Query 4: Open tickets — resolution_time_minutes is ABSENT
f_open = [
    {"left": {"metadataKey": "resolution_time_minutes"}, "operator": "NOT_EXISTS"},
]
r4 = retrieve("connector broken", filters=f_open)
show(r4, "FILTERED: resolution_time_minutes NOT_EXISTS (still open)")


## Step 11: Precision Lift — Side-by-Side

Same semantic query, one unfiltered, one with the operational filter. Counts and top-result relevance visible at a glance.


In [ ]:
print(f"Unfiltered 'payment issues':            {len(baseline)} results")
print(f"Filtered (billing + tier3):              {len(r1)} results")
print(f"Filtered (stuck tickets >120 min):       {len(r2)} results")
print(f"Filtered (recent auth):                  {len(r3)} results")
print(f"Filtered (still-open tickets):           {len(r4)} results")
print()
print("→ Metadata pre-filtering reduces the candidate set before KNN,")
print("  eliminating irrelevant-but-semantically-similar noise.")


## Step 12: Non-Semantic Enumeration

Sometimes you don't want similarity at all — you want every record matching a structured criterion. `list_memory_records` with metadata filters handles this: *"give me every stuck ticket this customer has"* — no vector search, deterministic enumeration.


In [ ]:
stuck_audit = data_client.list_memory_records(
    memoryId=memory_id,
    namespace=namespace,
    metadataFilters=[
        {"left": {"metadataKey": "resolution_time_minutes"}, "operator": "GREATER_THAN",
         "right": {"metadataValue": {"numberValue": 90}}}
    ],
)
for r in stuck_audit.get("memoryRecordSummaries", []):
    print(f"- {r.get('content', {}).get('text', '')[:180]}")


## Step 13: Live Agent with Metadata-Aware Context

Wire the filtered retrieval into a running Strands agent. The agent will be asked a question where the filtered context is meaningfully different from the unfiltered context.


In [ ]:
@tool
def lookup_refund_policy() -> str:
    """Internal tool: return the current refund policy."""
    return "Refunds of $500+ require supervisor approval. Processed in 3-5 business days."

# Configure the hook to retrieve with a billing + tier3 filter
hooks.set_context(
    handler="billing_specialist",
    product_area="billing",
    retrieve_filters=[
        {"left": {"metadataKey": "product_area"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "billing"}}},
        {"left": {"metadataKey": "escalation_chain"}, "operator": "CONTAINS",
         "right": {"metadataValue": {"stringValue": "tier3_engineer"}}},
    ],
)

support_agent = Agent(
    hooks=[hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[lookup_refund_policy],
    system_prompt=(
        "You are a senior SaaS support specialist. Use prior billing escalations that reached "
        "Tier-3 engineering as your primary reference for current billing escalations. "
        "Quote past root-causes and refund handling if applicable."
    ),
)

response = support_agent("I have another billing discrepancy this month — similar pattern to before.")
print("\n=== AGENT RESPONSE ===\n", response)


## Step 14: Additive Schema Evolution

Your schema will evolve. `update_memory` with `addIndexedMetadataKeys` adds new filterable dimensions without disturbing existing data. Existing records don't retroactively receive the new field, but as they undergo consolidation with newer ones, they naturally acquire it.

Note: you **cannot remove** an indexed key once added — this is intentional, preventing accidental loss of filtering capability on already-stored data.


In [ ]:
try:
    control_client.update_memory(
        memoryId=memory_id,
        addIndexedMetadataKeys=[
            {"metadataKey": "customer_segment", "metadataValueType": "STRING"}
        ],
    )
    logger.info("✅ Added customer_segment as a new indexed key")
except ClientError as e:
    logger.warning(f"update_memory: {e}")

# Confirm
updated = control_client.get_memory(memoryId=memory_id)["memory"]
print("Indexed keys now:")
for k in updated.get("indexedKeys", []):
    print(f"  - {k['key']} ({k['type']})")


## Step 15: Cleanup (Optional)

Uncomment to delete the memory resource.


In [ ]:
# control_client.delete_memory(memoryId=memory_id)
# print(f"Deleted {memory_id}")


## What you built

- A memory resource with **4 indexed keys** and a strategy-level `metadataSchema`.
- A custom `llmExtractionInstruction` that maintains an **append-only STRINGLIST** — one of the less-obvious extraction patterns.
- A non-indexed schema key (`customer_health_score`) that enriches records for reporting without consuming an indexed-key slot.
- Retrieval demos across **4 operator families**: `EQUALS_TO`, `CONTAINS`, `GREATER_THAN`, `NOT_EXISTS`, plus system-generated `AFTER`.
- A live Strands agent whose retrieval is scoped by metadata filters — so context injection is precise rather than lexically-fuzzy.
- Additive schema evolution via `update_memory`.

### Takeaways
- Metadata filtering turns retrieval from *"find something similar"* into *"find something similar **within this operational slice**"*.
- Custom merge instructions (append-only, hierarchy-based, monotonic) unlock memory semantics that simple `LATEST_VALUE` cannot express.
- Non-indexed schema keys let you enrich records without burning the 10-key budget.
- System-generated timestamp fields (`x-amz-agentcore-memory-createdAt`) give temporal filtering for free.
